# Demo: Building a Finance Q&A Agent using LangGraph

### Step 0: Install Required Libraries
<details>
<summary>📘 Explanation</summary>

- `langchain`: The core framework used to build chains and agents that connect LLMs with tools.
- `langchain-openai`: A specific integration that enables access to OpenAI and Azure OpenAI models.
</details>

In [ ]:
# Install LangChain and OpenAI integrations
!pip install langchain langchain-openai

### Step 1: Import Required Libraries and Set Up LLM

In [ ]:
import os
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate


# --- Azure OpenAI Setup ---
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"

llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net/",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"
)

def safe_python_repl(code: str) -> str:
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Error: {str(e)}"

class PythonREPL:
    def run(self, code):
        return safe_python_repl(code)

repl = PythonREPL()


### Step 2: Define the Prompt Template

In [ ]:
prompt = PromptTemplate.from_template(
    """You are a smart assistant for product managers. 
Given a business-related question, return:
1. Valid Python code using print(...) to calculate the answer.
2. A brief explanation of the result in plain English.

Question: {input}
Response:"""
)

### Step 3: Define the Helper Function

In [ ]:
def process_pm_query(q):
    response = llm.invoke(prompt.format(input=q)).content.strip()

    if "```python" in response:
        code_block = response.split("```python")[1].split("```")[0].strip()
    else:
        code_block = next((line for line in response.splitlines() if "print" in line), "")

    try:
        result = repl.run(code_block)
        return {
            "question": q,
            "code": code_block,
            "result": result.strip(),
            "explanation": response.replace(code_block, "").replace("```python", "").replace("```", "").strip()
        }
    except Exception as e:
        return {
            "question": q,
            "code": code_block,
            "error": str(e),
            "raw_response": response
        }

### Step 4: Run the Demo

In [ ]:
# --- Questions for PMs ---

questions = [
    "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?",
    "We lose 25% of our 40,000 users. How many do we retain?",
    "If each user pays $10/month and we have 5000 users, what is the monthly revenue?",
    "If revenue is $50,000 and cost is $37,000, what’s our profit?"
]

for q in questions:
    print(f"\nQuestion: {q}")
    output = process_pm_query(q)
    print("Code:", output.get("code", "No code"))
    print("Result:", output.get("result", output.get("error", "Failed to execute")))
    print("Explanation:", output.get("explanation", "No explanation"))